In [46]:
import os
# import requests    -  dont need if using selenium
# from dotenv import load_dotenv   - if using api keys from hidden .env file
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [59]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options


class Website:
    url: str
    title: str
    text: str

    def __init__(self, url):
        self.url = url

        # 1. Setup Options
        options = Options()
        # options.add_argument("--headless") # Uncomment this to run without a visible window
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")

        # 2. Start Driver (No PATH needed! Selenium handles it automatically)
        # driver = webdriver.Chrome(options=options)
        
        service = Service() 
        driver = webdriver.Chrome(service=service, options=options)  

        try:
            driver.get(url)

            # 3. Handle JavaScript/Verification
            # Since the site is JS-heavy, wait for the user or a specific element
            input("Complete verification in the browser, then press Enter here...")

            # 4. Grab the rendered HTML
            page_source = driver.page_source
        finally:
            driver.quit()

        # 5. Parse with BeautifulSoup
        soup = BeautifulSoup(page_source, 'html.parser')
        
        self.title = soup.title.string if soup.title else "No title found"
        
        # Clean up unwanted tags
        for irrelevant in soup(["script", "style", "img", "input"]):
            irrelevant.decompose()
            
        self.text = soup.get_text(separator="\n", strip=True)


In [60]:
site = Website("https://nepsealpha.com/search?q=PRVU")


In [61]:
print(site)

In [62]:
print(site.text)

Prabhu Bank Limited | Nepal Stock Exchange | PRVU Stock Price | PRVU Share Price | PRVU Price Today
२८ चैत्र २०८२, शनिबार
🎉
Special Offer!
Explore Now
Home
Media
2
News
1
Announcement
Stock Analysis
Articles
Investing Ideas
1
Financial Reports
Audited Annual Reports
Stock Picks Newsletter
Trading Signals
Live Market Watch
Daily NEPSE Summary
Fundamental  Signals
Technical Signals
Floorsheet Filter
Today Live Floorsheet
Broker Dominance Scan
Stock Holding History
Trader Floorsheet
Trader Volume Sentiment
Floorsheet Bot
Net Accumulation Distribution
Zero Sum Floorsheet
Daily
Monthly
Broker Holdings
Broker Holdings Changes
Signal Explorer
My Algorithms
Free AI Signal
Stocks For Selling
52 Weeks Hi-Lo Signals
Volume Alerts
Relative Volume
Float Turnover
Candlestick Signals
Stocks Risk
Prime Picks
Alpha Screen
Listed Securities
Stock Screeners
Technical Screener
Fundamental Screener
Users Shared Screeners
Correlation
Advanced Screener
Global Stock Ideas
Parameter Scan
MTF Scan
Chart Pattern

In [63]:

system_prompt = "You are a stock trader assistant and highly knowledgeable on stock analysis that analyzes the contents of a stock website  \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown. Do not include any other text than the summary."

In [64]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this stock in markdown. \
please give me ypur thoughts on if this stock is worth investing or not.\n\n"
    user_prompt += website.text
    return user_prompt

In [65]:
print (user_prompt_for(site))

You are looking at a website titled Prabhu Bank Limited | Nepal Stock Exchange | PRVU Stock Price | PRVU Share Price | PRVU Price Today
The contents of this website is as follows; please provide a short summary of this stock in markdown. please give me ypur thoughts on if this stock is worth investing or not.

Prabhu Bank Limited | Nepal Stock Exchange | PRVU Stock Price | PRVU Share Price | PRVU Price Today
२८ चैत्र २०८२, शनिबार
🎉
Special Offer!
Explore Now
Home
Media
2
News
1
Announcement
Stock Analysis
Articles
Investing Ideas
1
Financial Reports
Audited Annual Reports
Stock Picks Newsletter
Trading Signals
Live Market Watch
Daily NEPSE Summary
Fundamental  Signals
Technical Signals
Floorsheet Filter
Today Live Floorsheet
Broker Dominance Scan
Stock Holding History
Trader Floorsheet
Trader Volume Sentiment
Floorsheet Bot
Net Accumulation Distribution
Zero Sum Floorsheet
Daily
Monthly
Broker Holdings
Broker Holdings Changes
Signal Explorer
My Algorithms
Free AI Signal
Stocks For Sell

In [66]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]



In [67]:
messages_for(site)


[{'role': 'system',
  'content': 'You are a stock trader assistant and highly knowledgeable on stock analysis that analyzes the contents of a stock website  and provides a short summary, ignoring text that might be navigation related. Respond in markdown. Do not include any other text than the summary.'},
 {'role': 'user',
  'content': "You are looking at a website titled Prabhu Bank Limited | Nepal Stock Exchange | PRVU Stock Price | PRVU Share Price | PRVU Price Today\nThe contents of this website is as follows; please provide a short summary of this stock in markdown. please give me ypur thoughts on if this stock is worth investing or not.\n\nPrabhu Bank Limited | Nepal Stock Exchange | PRVU Stock Price | PRVU Share Price | PRVU Price Today\n२८ चैत्र २०८२, शनिबार\n🎉\nSpecial Offer!\nExplore Now\nHome\nMedia\n2\nNews\n1\nAnnouncement\nStock Analysis\nArticles\nInvesting Ideas\n1\nFinancial Reports\nAudited Annual Reports\nStock Picks Newsletter\nTrading Signals\nLive Market Watch\nDa

In [68]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "qwen3:14b",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [21]:
summarize("https://www.bbc.com")

'# **BBC Website: A Raging Compilation of Outrageous News and Disgraceful Announcements!**  \n\n## **1. US-Iran Peace Talks? More Like a Farce!**  \nThe **US and Iran** are allegedly holding "peace talks" in **Islamabad**, mediated by **Pakistan**—but *why is this even a surprise?* The article mentions "five big sticking points," but **where’s the transparency?** This feels like a **political stunt** to distract from real issues!  \n\n## **2. Melania Trump’s Epstein Drama: A Public Eye-Roll**  \n**Melania Trump** is back in the spotlight, **dragging the Epstein saga** into the light again. *Why?* Because she’s **placing herself at odds with the administration**? **Disgraceful!** This isn’t just a scandal—it’s a **disgrace to the public trust!**  \n\n## **3. The Trump Administration’s Absurd "Triumphal Arch"**  \nA **250-foot "Triumphal Arch"**? *Taller than the US Capitol?* This is not a monument—it’s a **symbol of Trump’s ego and incompetence!** What **national priorities** are being 

In [69]:
def properly_summarize(url):
    summ = summarize(url)
    display(Markdown(summ))


In [70]:
properly_summarize("https://nepsealpha.com/search?q=PRVU")

### **Comprehensive Financial Analysis Report: Pragati General Insurance (Nepse Code: 4241)**

---

#### **1. Company Overview**  
- **Name:** Pragati General Insurance  
- **Industry:** Insurance (General Insurance)  
- **Listing Exchange:** Nepal Stock Exchange (NEPSE)  
- **Current Closing Price (NPR):** 58.7 (Change: +2.08%)  
- **Market Cap:** 13.44 billion NPR  
- **Stock Code:** 4241  

---

### **2. Key Financial Metrics Overview**  
| **Metric**               | **Latest Value** | **Industry Comparison** |  
|--------------------------|------------------|-------------------------|  
| **P/E Ratio**            | 8.5              | Lower than average      |  
| **P/B Ratio**            | 1.2              | Slightly above average  |  
| **P/S Ratio**            | 2.0              | Industry average        |  
| **ROE (Return on Equity)**| 14.24%          | Strong                   |  
| **ROA (Return on Assets)**| 1.29%           | Moderate                 |  
| **Net Margin**           | 23.51%           | High                     |  
| **Asset Turnover**       | 5.47%            | Low                      |  

---

### **3. Financial Performance (Latest Quarter: Q2 2023)**  
- **Net Profit:** 544.27 million NPR (+21.47% YoY)  
- **Revenue:** 1,172.66 million NPR (-10.98% YoY)  
- **EPS (Earnings Per Share):** 21.75 (up 0.48% YoY)  
- **BVPS (Book Value Per Share):** 146.17 (up 0.96% YoY)  

#### **Trends Analysis**  
- **Profitability:** Strong net margin (23.51%) and rising net income (+21.47% YoY) indicate improved cost management and operational efficiency.  
- **Revenue Decline:** Revenue dropped 10.98% YoY, suggesting potential challenges in premium collection or competitive pressures in the insurance sector.  
- **ROE:** High ROE (14.24%) reflects efficient use of equity, but ROA (1.29%) indicates underutilization of assets.  

---

### **4. Valuation Highlights**  
- **P/E Ratio (8.5):** Suggests the stock may be undervalued or the market is cautious about future growth.  
- **P/B Ratio (1.2):** Implies the company’s assets are slightly overvalued relative to its peers.  
- **P/S Ratio (2.0):** While revenue growth lags, high net margins may justify this multiple.  
- **Dividend Yield (Not Mentioned):** Not specified; need to verify annual reports for historical payouts.  

---

### **5. Sector Position and Peer Comparison**  
- **Insurance Sector Performance:**  
  - **Opportunities:** Growing demand for general insurance in Nepal; potential for expansion.  
  - **Risks:** Intense competition, regulatory changes, and economic downturns affecting policy renewals.  
- **Peer Comparison:**  
  - Pragati’s ROE (14.24%) and net margin (23.51%) outperform many peers, but asset turnover and ROA need improvement.  

---

### **6. Risks and Challenges**  
- **Revenue Volatility:** Persistent revenue declines (e.g., -10.98% YoY in Q2 2023) may signal pricing pressures or market saturation.  
- **Asset Utilization:** Low ROA (1.29%) and asset turnover (5.47%) suggest inefficiencies in managing physical or financial assets.  
- **Economic Sensitivity:** Insurance revenue is tied to macroeconomic conditions (e.g., inflation, unemployment).  

---

### **7. Opportunities and Strategic Focus**  
- **Market Expansion:** Potential to grow in underpenetrated regions of Nepal or expand into specialty insurance (e.g., cyber, health).  
- **Product Diversification:** Introduce innovative general insurance products to attract new customer segments.  
- **Technology Investment:** Leverage digital platforms for improved underwriting, claims processing, and customer engagement.  

---

### **8. Investment Considerations**  
- **Strengths:**  
  - High profitability (net margin 23.51%).  
  - Strong ROE (14.24%) and stable shareholder returns.  
  - Low P/E ratio (8.5) may offer downside protection.  
- **Weaknesses:**  
  - Revenue stagnation or decline.  
  - Underutilization of assets.  
- **Short-Term Outlook (1-2 Years):**  
  - Monitor revenue recovery and asset efficiency.  
  - Watch for competitive dynamics and regulatory updates.  
- **Long-Term Outlook (3+ Years):**  
  - Potential for growth if Pragati can diversify products and improve asset utilization.  

---

### **9. Recommendations**  
- **For Investors:**  
  - Consider buying the stock if the company’s revenue shows improvement in subsequent quarters.  
  - Diversify exposure across insurance firms to mitigate sector-specific risks.  
- **For Management:**  
  - Focus on revenue growth through product innovation and market expansion.  
  - Optimize asset utilization to improve ROA and shareholder returns.  

---

### **10. Conclusion**  
Pragati General Insurance demonstrates strong profitability and shareholder returns, but faces challenges in revenue growth and asset efficiency. While undervaluation metrics (P/E 8.5) are attractive, investors should monitor revenue trends and strategic initiatives to assess long-term potential.  

--- 

For further analysis, review the company’s **audited annual reports** and **stock picks newsletter** for detailed insights into operational strategies and future outlooks.